# **RICELEAF DISEASE PREDICTION**

The dataset contains 120 images of rice leaves categorized into three disease classes:

*   Leaf smut
*   Brown spot
*   Bacterial leaf blight

Each class contains 40 images, making the dataset balanced.

In [ ]:
from google.colab import files
files.upload()

In [ ]:
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

img = image.load_img("cnn/Leaf smut/DSC_0308.JPG")
plt.imshow(img)
plt.axis('off')

In [ ]:
import zipfile

with zipfile.ZipFile("CNN.zip", 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
import os

os.listdir()

In [ ]:
os.listdir('cnn')

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

img_size = (150, 150)
batch_size = 10

#Our data is too small for training the model, so using data augmentation to fake more datas
train_datagen = ImageDataGenerator(
    rescale=1./255,          # normalize pixel values
    validation_split=0.2,    # validation data (used during training)

    rotation_range=30,       # rotate image
    zoom_range=0.3,          # zoom in/out
    horizontal_flip=True    # flip image
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    "cnn",
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_data = val_datagen.flow_from_directory(
    "cnn",
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# **DATA AUGMENTATION**

Data augmentation is used to artificially increase the size and diversity of the dataset by applying transformations to existing images.

The following augmentation techniques were applied using ImageDataGenerator:


*   Rescaling (1./255): Normalizes pixel values from 0–255 to 0–1, helping the model learn efficiently

*   Validation Split (0.2): Reserves 20% of data for validation during training.

*   Rotation (10 degrees): Slightly rotates images to simulate different orientations.
*   Zoom (0.1): Zooms images in and out to create variation.


*   Horizontal Flip: Flips images to increase diversity

Importance:

*   The dataset is very small (only 120 images)

*   It helps reduce overfitting
*   It improves model generalization


*   It simulates real-world variations in leaf images

Here, in this project, Data augmentation improves model performance by exposing the model to multiple variations of the same image, thereby helping it learn more robust and generalized features.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping  #EarlyStopping ensures that the model retains the weights from the epoch with the lowest validation loss.

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_data.classes),
    y=train_data.classes
)

class_weights = dict(enumerate(class_weights))

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input

model = Sequential([
    Input(shape=(150,150,3)),

    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(),

    Flatten(),

    Dense(64, activation='relu'),
    Dropout(0.5),

    Dense(3, activation='softmax')
])

In [ ]:
# Model training
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,
    class_weight=class_weights,
    callbacks=[early_stop]
)

In [ ]:
# Model structure check

model.summary()

In [ ]:
# Accuracy
plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='val accuracy')
plt.legend()
plt.title("Accuracy")
plt.show()

# Loss
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.legend()
plt.title("Loss")
plt.show()

In [ ]:
# Both training and validation accuracy improved while loss decreased, indicating effective learning and better model generalization

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

y_pred = model.predict(val_data)
y_pred_classes = np.argmax(y_pred, axis=1)

y_true = val_data.classes

print(classification_report(y_true, y_pred_classes))

In [ ]:
# Test with single image

In [ ]:
os.listdir("cnn")

In [ ]:
os.listdir("cnn/Bacterial leaf blight")

In [ ]:
from tensorflow.keras.preprocessing import image

img_path = "cnn/Bacterial leaf blight/DSC_0377.JPG"

img = image.load_img(img_path, target_size=(150,150))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

classes = list(train_data.class_indices.keys())
print("Prediction:", classes[np.argmax(prediction)])

Challenges:



*   Small dataset size (120 images)
*   High similarity between disease patterns

*   Risk of overfitting